# Notebook 2 of 3 — Generation (manual §5.4 step 5, Tasks 2 and 6)

**Settings:** Accelerator **GPU T4 x2**, Internet **On**.
**Input:** Add Input → Your Work → Notebooks → the saved version of **Notebook 1 (training)**.

| Generation | Eval set | Purpose |
|---|---|---|
| Qwen2.5-0.5B base | AlpacaEval 300 (seed 42) | Task 2 baseline |
| SFT best, DPO-UF best | AlpacaEval 300 (seed 42) | UltraFeedback DPO vs SFT |
| SFT best, DPO-HH best | provided HH helpful/harmless set (300) | HH-RLHF DPO vs SFT |
| SmolLM2-360M-Instruct, + LoRA adapter | persona style set (40) | LoRA persona evaluation |

Greedy decoding, identical settings for both models in each comparison. Each file is written as soon as its
model finishes; pairwise JSONL files are written before any judging (manual §5.5).

In [1]:
# Manual section 7.1: clone the repository in Kaggle (Notebook settings: Internet ON).
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

/kaggle/working/cs5326-PA2-27100015
6a2ff79 (HEAD -> main, origin/main, origin/HEAD) Add training results from Kaggle run of notebook 1


In [2]:
import subprocess
def sh(cmd):
    """Run a shell command, show its output in this cell, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} (see output above): {cmd}")

In [3]:
# Manual sections 5.2 and 7.2: install dependencies without reinstalling PyTorch.
# peft/trl are added for the Task 5 LoRA run; torchao is removed to avoid the known PEFT conflict.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl>=0.24,<0.30"
!python -m pip install -q -e . --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 961.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) 

In [4]:
# Locate files from the attached input dataset (Add Input -> the saved version of the previous notebook).
import glob
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the saved notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [5]:
SFT = os.path.dirname(find_input("sft/best/checkpoint_info.json"))
DPO_HH = os.path.dirname(find_input("dpo_hh/best/checkpoint_info.json"))
DPO_UF = os.path.dirname(find_input("dpo_uf/best/checkpoint_info.json"))
LORA = os.path.dirname(find_input("lora_dpo/final/adapter_config.json"))
BASE, SMOL = "Qwen/Qwen2.5-0.5B", "HuggingFaceTB/SmolLM2-360M-Instruct"
ALPACA, HH, PERSONA = "eval_sets/alpaca_eval_300_seed42.json", "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/persona_style_eval.json"
G = "outputs/generations"
os.makedirs(G, exist_ok=True)

sft/best/checkpoint_info.json -> /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/sft/best/checkpoint_info.json
dpo_hh/best/checkpoint_info.json -> /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/dpo_hh/best/checkpoint_info.json
dpo_uf/best/checkpoint_info.json -> /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/dpo_uf/best/checkpoint_info.json
lora_dpo/final/adapter_config.json -> /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/lora_dpo/final/adapter_config.json


### Decoding settings
Greedy decoding by default (§8). If the sanity check shows repetitive loops, set a modest repetition penalty
(e.g. 1.1) here, re-run from this cell, and document the choice in the report.

In [6]:
DECODE = "--max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0"  # same for every model in a comparison

## Sanity check on 3 prompts per checkpoint (§12): path, tokenizer, prompt format, EOS, length

In [7]:
sh(f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base --eval_set {ALPACA} --output {G}/sanity_base.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {ALPACA} --output {G}/sanity_sft.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {DPO_UF} --generator dpo_uf --eval_set {ALPACA} --output {G}/sanity_dpo_uf.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {DPO_HH} --generator dpo_hh --eval_set {HH} --output {G}/sanity_dpo_hh.json --limit 3 {DECODE}")
sh(f"python scripts/generate.py --model_path {SMOL} --adapter_path {LORA} --prompt_format chat_template --generator smollm2-lora-dpo --eval_set {PERSONA} --output {G}/sanity_lora.json --limit 3 --max_new_tokens 256")

$ python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/sanity_base.json --limit 3 --max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0
`torch_dtype` is deprecated! Use `dtype` instead!
3 prompts, 0 already generated, 3 to go
  3/3
INSTRUCTION: Create a short, concise summary of the paper based on its abstract.

Few-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks: intent detect 
--- OUTPUT:
 You are a helpful assistant.Jeste
Create a short, concise summary of the paper based on its abstract.

Few-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks

## Task 2 — Baseline generation: Qwen2.5-0.5B base on the fixed AlpacaEval subset

In [8]:
sh(f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base --eval_set {ALPACA} --output {G}/alpaca_base.json {DECODE}")

$ python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/alpaca_base.json --max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0
`torch_dtype` is deprecated! Use `dtype` instead!
300 prompts, 0 already generated, 300 to go
  16/300
  32/300
  48/300
  64/300
  80/300
  96/300
  112/300
  128/300
  144/300
  160/300
  176/300
  192/300
  208/300
  224/300
  240/300
  256/300
  272/300
  288/300
  300/300


## Task 6 — Evaluation generations (two models at a time, one per GPU)

In [9]:
# UltraFeedback track: SFT vs DPO-UF on AlpacaEval (same seed-42 subset as the baseline).
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {ALPACA} --output {G}/alpaca_sft.json --device cuda:0 {DECODE} > {G}/log_alpaca_sft.txt 2>&1 & "
   f"python scripts/generate.py --model_path {DPO_UF} --generator dpo_uf --eval_set {ALPACA} --output {G}/alpaca_dpo_uf.json --device cuda:1 {DECODE} > {G}/log_alpaca_dpo_uf.txt 2>&1 & wait")

$ python scripts/generate.py --model_path /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/sft/best --generator sft --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/alpaca_sft.json --device cuda:0 --max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0 > outputs/generations/log_alpaca_sft.txt 2>&1 & python scripts/generate.py --model_path /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/dpo_uf/best --generator dpo_uf --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/alpaca_dpo_uf.json --device cuda:1 --max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0 > outputs/generations/log_alpaca_dpo_uf.txt 2>&1 & wait


In [10]:
# HH-RLHF track: SFT vs DPO-HH on the provided synthetic helpful/harmless set.
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {HH} --output {G}/hh_sft.json --device cuda:0 {DECODE} > {G}/log_hh_sft.txt 2>&1 & "
   f"python scripts/generate.py --model_path {DPO_HH} --generator dpo_hh --eval_set {HH} --output {G}/hh_dpo_hh.json --device cuda:1 {DECODE} > {G}/log_hh_dpo_hh.txt 2>&1 & wait")

$ python scripts/generate.py --model_path /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/sft/best --generator sft --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output outputs/generations/hh_sft.json --device cuda:0 --max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0 > outputs/generations/log_hh_sft.txt 2>&1 & python scripts/generate.py --model_path /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/dpo_hh/best --generator dpo_hh --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output outputs/generations/hh_dpo_hh.json --device cuda:1 --max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0 > outputs/generations/log_hh_dpo_hh.txt 2>&1 & wait


In [11]:
# LoRA persona track: SmolLM2-360M-Instruct with and without the LoRA-DPO adapter (its own chat template).
PERSONA_DECODE = "--max_new_tokens 256 --batch_size 16 --repetition_penalty 1.0"
sh(f"python scripts/generate.py --model_path {SMOL} --prompt_format chat_template --generator smollm2-instruct --eval_set {PERSONA} --output {G}/persona_base.json --device cuda:0 {PERSONA_DECODE} > {G}/log_persona_base.txt 2>&1 & "
   f"python scripts/generate.py --model_path {SMOL} --adapter_path {LORA} --prompt_format chat_template --generator smollm2-lora-dpo --eval_set {PERSONA} --output {G}/persona_lora.json --device cuda:1 {PERSONA_DECODE} > {G}/log_persona_lora.txt 2>&1 & wait")

$ python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --prompt_format chat_template --generator smollm2-instruct --eval_set eval_sets/persona_style_eval.json --output outputs/generations/persona_base.json --device cuda:0 --max_new_tokens 256 --batch_size 16 --repetition_penalty 1.0 > outputs/generations/log_persona_base.txt 2>&1 & python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path /kaggle/input/notebooks/mahama5if/notebbok1-genaipa2/cs5326-PA2-27100015/outputs/lora_dpo/final --prompt_format chat_template --generator smollm2-lora-dpo --eval_set eval_sets/persona_style_eval.json --output outputs/generations/persona_lora.json --device cuda:1 --max_new_tokens 256 --batch_size 16 --repetition_penalty 1.0 > outputs/generations/log_persona_lora.txt 2>&1 & wait


In [12]:
# Parallel runs report success even if one process failed, so verify every file is complete.
import json
expected = {"alpaca_base": ALPACA, "alpaca_sft": ALPACA, "alpaca_dpo_uf": ALPACA, "hh_sft": HH, "hh_dpo_hh": HH,
            "persona_base": PERSONA, "persona_lora": PERSONA}
for name, eval_set in expected.items():
    path = f"{G}/{name}.json"
    assert os.path.exists(path), f"{path} missing: check {G}/log_{name}.txt"
    g, n = json.load(open(path)), len(json.load(open(eval_set)))
    assert len(g) == n, f"{path} has {len(g)}/{n} outputs: re-run its cell (it resumes) and check the log"
    print(f"{name}: {len(g)} outputs, {sum(x['hit_max_new_tokens'] for x in g)} hit max_new_tokens, "
          f"mean chars {sum(len(x['output']) for x in g) / len(g):.0f}")

alpaca_base: 300 outputs, 201 hit max_new_tokens, mean chars 1637
alpaca_sft: 300 outputs, 49 hit max_new_tokens, mean chars 875
alpaca_dpo_uf: 300 outputs, 78 hit max_new_tokens, mean chars 1307
hh_sft: 300 outputs, 44 hit max_new_tokens, mean chars 992
hh_dpo_hh: 300 outputs, 23 hit max_new_tokens, mean chars 756
persona_base: 40 outputs, 17 hit max_new_tokens, mean chars 715
persona_lora: 40 outputs, 30 hit max_new_tokens, mean chars 885


## Pairwise comparison files (§12): randomized A/B order, model identity kept as hidden metadata

In [13]:
sh(f"python scripts/build_pairs.py --candidate {G}/alpaca_dpo_uf.json --candidate_name dpo_uf --baseline {G}/alpaca_sft.json --baseline_name sft --output outputs/eval/dpo_uf_vs_sft_pairs.jsonl")
sh(f"python scripts/build_pairs.py --candidate {G}/hh_dpo_hh.json --candidate_name dpo_hh --baseline {G}/hh_sft.json --baseline_name sft --output outputs/eval/dpo_hh_vs_sft_pairs.jsonl")
sh(f"python scripts/build_pairs.py --candidate {G}/persona_lora.json --candidate_name lora_dpo --baseline {G}/persona_base.json --baseline_name smollm2_instruct --output outputs/eval/lora_vs_base_pairs.jsonl")

$ python scripts/build_pairs.py --candidate outputs/generations/alpaca_dpo_uf.json --candidate_name dpo_uf --baseline outputs/generations/alpaca_sft.json --baseline_name sft --output outputs/eval/dpo_uf_vs_sft_pairs.jsonl
wrote 300 pairs to outputs/eval/dpo_uf_vs_sft_pairs.jsonl (candidate in position A for 151)
$ python scripts/build_pairs.py --candidate outputs/generations/hh_dpo_hh.json --candidate_name dpo_hh --baseline outputs/generations/hh_sft.json --baseline_name sft --output outputs/eval/dpo_hh_vs_sft_pairs.jsonl
wrote 300 pairs to outputs/eval/dpo_hh_vs_sft_pairs.jsonl (candidate in position A for 151)
$ python scripts/build_pairs.py --candidate outputs/generations/persona_lora.json --candidate_name lora_dpo --baseline outputs/generations/persona_base.json --baseline_name smollm2_instruct --output outputs/eval/lora_vs_base_pairs.jsonl
wrote 40 pairs to outputs/eval/lora_vs_base_pairs.jsonl (candidate in position A for 17)


In [14]:
!cd outputs && zip -q -r /kaggle/working/generation_outputs.zip generations eval
!ls -lh /kaggle/working/generation_outputs.zip

-rw-r--r-- 1 root root 725K Oct 18 10:00 /kaggle/working/generation_outputs.zip


**Done.** This version's outputs are the input dataset for Notebook 3 (evaluation).